# Notebook 10 — SCRFD Face Scale & Canvas Diagnostic

**One-click Colab Run all.** Uses frozen 14 AffectNet+ sources and the SHA256-verified N09 detector. No diffusion, identity embeddings, or privacy claims. All acceptance checks remain pending until visual review. Python 3.12 on Colab required.

## 1. Frozen scientific settings and source manifest

Five predeclared input/canvas conditions isolate resizing vs. padding. Threshold stays fixed at 0.10; no adaptive optimization.

In [ ]:
import json, hashlib, os, sys, platform, subprocess, tempfile, time, uuid, csv
from pathlib import Path
from datetime import datetime, timezone
CONFIG = json.loads("{\n  \"schema_version\": 1,\n  \"method_id\": \"nb10_detector_scale_canvas_v1\",\n  \"classification\": \"exploratory_scientific_validity_diagnostic\",\n  \"parent07\": \"20260818T210720Z_notebook07_affectnet_larger_6aeb3643_f1142664\",\n  \"parent08\": \"20261013_nb08_biometric-diagnostic_48369686_3f72a02e_20261013T181313795949Z_eb90117f\",\n  \"sources\": [\n    {\n      \"source_id\": \"source_000\",\n      \"emotion\": \"neutral\",\n      \"historical_stratum\": \"success\",\n      \"source_sha256\": \"9aaf998afb4add466083d80efdf02e861defb67a5d20920c1087edcb7a0a2f5e\",\n      \"archive_source_sha256\": \"5821f92d36917acf7b48f6ff73fc74b906f49deef6f279e126fb6d33f338c5df\",\n      \"archive_member\": \"AffectNet+/human_annotated/validation_set/images/2520.jpg\"\n    },\n    {\n      \"source_id\": \"source_001\",\n      \"emotion\": \"neutral\",\n      \"historical_stratum\": \"faceanalysis_failed\",\n      \"source_sha256\": \"fb2d29c2efc0a05f547a7214ac8b06a569d720d746a919e0cfaeba7483eda3c0\",\n      \"archive_source_sha256\": \"b61a8c328affd7df7da62b9a94d44057e2a87757953541fde502391b9e38f28b\",\n      \"archive_member\": \"AffectNet+/human_annotated/validation_set/images/4797.jpg\"\n    },\n    {\n      \"source_id\": \"source_005\",\n      \"emotion\": \"happy\",\n      \"historical_stratum\": \"success\",\n      \"source_sha256\": \"9b447fe3e03df12c2e3127b73da3e47fcbe0e4a711e218db572543c80de3754a\",\n      \"archive_source_sha256\": \"2a3394314baba82c49e568574d98dac1ed08a8b6611f571b17dcfd823cd3e451\",\n      \"archive_member\": \"AffectNet+/human_annotated/validation_set/images/3189.jpg\"\n    },\n    {\n      \"source_id\": \"source_008\",\n      \"emotion\": \"happy\",\n      \"historical_stratum\": \"faceanalysis_failed\",\n      \"source_sha256\": \"d6093914c41eb42f8857fb73c078316a93440fc0a2e94e021d363c1f6ac1435d\",\n      \"archive_source_sha256\": \"e29952d0fffc48145966873e2c43611d36ff92ff8cafb1810741009922d46af5\",\n      \"archive_member\": \"AffectNet+/human_annotated/validation_set/images/3346.jpg\"\n    },\n    {\n      \"source_id\": \"source_013\",\n      \"emotion\": \"sad\",\n      \"historical_stratum\": \"success\",\n      \"source_sha256\": \"dfe31cea4d43247c28a7f72cc44954a74c962fe1bbc869c27b42a4be689de6b9\",\n      \"archive_source_sha256\": \"88de72b2757b80d4b9e86ab34d746ae114e53cc03a6eb8b9484a21c2c93b7881\",\n      \"archive_member\": \"AffectNet+/human_annotated/validation_set/images/1711.jpg\"\n    },\n    {\n      \"source_id\": \"source_010\",\n      \"emotion\": \"sad\",\n      \"historical_stratum\": \"faceanalysis_failed\",\n      \"source_sha256\": \"662b49c4ff8afb7ba9ee7de03002706362b491145b161b54622b3cf648a7fd7e\",\n      \"archive_source_sha256\": \"53a1079c082bbd1bacdbc942f756c912936e8fc6fd89692e6017fc51612dc09c\",\n      \"archive_member\": \"AffectNet+/human_annotated/validation_set/images/2633.jpg\"\n    },\n    {\n      \"source_id\": \"source_017\",\n      \"emotion\": \"surprise\",\n      \"historical_stratum\": \"success\",\n      \"source_sha256\": \"86273e10c39c5d9f52c1d79ef323eef7b3d6bcbbfbb3ca78782b39d34adf9578\",\n      \"archive_source_sha256\": \"1b144704e399f25c700e7767ec09984197ca9075b3036a3ca62b55c59dfb1406\",\n      \"archive_member\": \"AffectNet+/human_annotated/validation_set/images/2907.jpg\"\n    },\n    {\n      \"source_id\": \"source_015\",\n      \"emotion\": \"surprise\",\n      \"historical_stratum\": \"faceanalysis_failed\",\n      \"source_sha256\": \"83e85260fccf7a1295116255b4f33c6672a26463941db0ddd3c71970e3a1b884\",\n      \"archive_source_sha256\": \"3e224e2df82d9df37166ce729850b23a268828d2ee18184664d567c1f25926a7\",\n      \"archive_member\": \"AffectNet+/human_annotated/validation_set/images/5338.jpg\"\n    },\n    {\n      \"source_id\": \"source_021\",\n      \"emotion\": \"fear\",\n      \"historical_stratum\": \"success\",\n      \"source_sha256\": \"77b9b9002553c6d020f6644ac1bde02a14466f00080841e337f58238b4166cc3\",\n      \"archive_source_sha256\": \"fb51430ae7cfe88b880d6c46f08712ecd5aa588a5e9cb62ea480c8212b22716a\",\n      \"archive_member\": \"AffectNet+/human_annotated/validation_set/images/1943.jpg\"\n    },\n    {\n      \"source_id\": \"source_020\",\n      \"emotion\": \"fear\",\n      \"historical_stratum\": \"faceanalysis_failed\",\n      \"source_sha256\": \"58a93b846390c860b9a5dbf4732d637c8aafe5f1993a46770d5db12e11b1cd62\",\n      \"archive_source_sha256\": \"9a68fcc282c18426252d401ffeffd6a583ec9bb82b2d65493c70f314e1f2ac7a\",\n      \"archive_member\": \"AffectNet+/human_annotated/validation_set/images/1762.jpg\"\n    },\n    {\n      \"source_id\": \"source_026\",\n      \"emotion\": \"angry\",\n      \"historical_stratum\": \"success\",\n      \"source_sha256\": \"6c119acab8154815cf942acbe301604abf40926feb3c4d76d98b8f4c690c314f\",\n      \"archive_source_sha256\": \"1e36948076761c7cf56665ff94d11716ce5f60b6ac8ca578dacb596197b019dd\",\n      \"archive_member\": \"AffectNet+/human_annotated/validation_set/images/2451.jpg\"\n    },\n    {\n      \"source_id\": \"source_025\",\n      \"emotion\": \"angry\",\n      \"historical_stratum\": \"faceanalysis_failed\",\n      \"source_sha256\": \"d19344378aafe06f583b2a0a66ed2ec52c13d9db734fbd9881746c61a52b467c\",\n      \"archive_source_sha256\": \"b5cd76ba679f7655ed6e65b05a5d95f55f3e137aaceab3c25b28b385e5c9ea44\",\n      \"archive_member\": \"AffectNet+/human_annotated/validation_set/images/3479.jpg\"\n    },\n    {\n      \"source_id\": \"source_044\",\n      \"emotion\": \"neutral\",\n      \"historical_stratum\": \"yolo_failed\",\n      \"source_sha256\": \"fc6decda2edbe4099da823869d0c79b2c9f97592892aa9d27869f5fcc518977b\",\n      \"archive_source_sha256\": \"9c10a24c566f19f0d7500c69948998f01565a304151a3613952720790249b7d3\",\n      \"archive_member\": \"AffectNet+/human_annotated/validation_set/images/3473.jpg\"\n    },\n    {\n      \"source_id\": \"source_019\",\n      \"emotion\": \"surprise\",\n      \"historical_stratum\": \"yolo_failed\",\n      \"source_sha256\": \"ac565e56517b1bea481dbc5bde6c21ac604183ae3866ddc8f9db72c8eea90d8e\",\n      \"archive_source_sha256\": \"9569b489d4d71b364d64275bccd839ca206f9423beccea5c8528e3f45037f384\",\n      \"archive_member\": \"AffectNet+/human_annotated/validation_set/images/3385.jpg\"\n    }\n  ],\n  \"priority\": [\n    \"source_008\",\n    \"source_015\",\n    \"source_017\",\n    \"source_000\",\n    \"source_013\",\n    \"source_005\",\n    \"source_021\",\n    \"source_026\"\n  ],\n  \"references\": {\n    \"R1\": \"Recomputed N08 A: RGB ndarray YOLO largest box/margin15; compare saved prepared hash; FaceAnalysis 1.0.1 det1024 thresh0.1\",\n    \"R2\": \"Minimal upstream FaceAnalysis full RGB prepared image -> BGR; same models, det1024 thresh0.1; direct detect/get/trace comparison\",\n    \"R3\": \"Independent YOLO face graph on full PIL RGB input, confidence0.25; diagnostic only\",\n    \"R4\": \"Conditional legacy 0.7.3 decoder replay of identical captured raw tensors only if independent decode disagrees with native coordinates\"\n  },\n  \"det_size\": [\n    1024,\n    1024\n  ],\n  \"det_thresh\": 0.1,\n  \"yolo_confidences\": [\n    0.25,\n    0.1,\n    0.05\n  ],\n  \"crop_margin\": 0.15,\n  \"seed\": 42,\n  \"providers\": [\n    \"CPUExecutionProvider\"\n  ],\n  \"device\": \"cpu\",\n  \"models\": {\n    \"arcface\": {\n      \"pack\": \"buffalo_l\",\n      \"archive_url\": \"https://github.com/deepinsight/insightface/releases/download/v0.7/buffalo_l.zip\",\n      \"onnx_sha256\": {\n        \"1k3d68.onnx\": \"df5c06b8a0c12e422b2ed8947b8869faa4105387f199c477af038aa01f9a45cc\",\n        \"2d106det.onnx\": \"f001b856447c413801ef5c42091ed0cd516fcd21f2d6b79635b1e733a7109dbf\",\n        \"det_10g.onnx\": \"5838f7fe053675b1c7a08b633df49e7af5495cee0493c7dcf6697200b85b5b91\",\n        \"genderage.onnx\": \"4fde69b1c810857b88c64a335084f1c3fe8f01246c9a191b48c7bb756d6652fb\",\n        \"w600k_r50.onnx\": \"4c06341c33c2ca1f86781dab0e829f88ad5b64be9fba56e56bc9ebdefc619e43\"\n      }\n    },\n    \"auraface\": {\n      \"repo\": \"fal/AuraFace-v1\",\n      \"revision\": \"686774cad65e40933022896195e07e01ee06bee9\",\n      \"onnx_sha256\": {\n        \"1k3d68.onnx\": \"df5c06b8a0c12e422b2ed8947b8869faa4105387f199c477af038aa01f9a45cc\",\n        \"2d106det.onnx\": \"f001b856447c413801ef5c42091ed0cd516fcd21f2d6b79635b1e733a7109dbf\",\n        \"genderage.onnx\": \"4fde69b1c810857b88c64a335084f1c3fe8f01246c9a191b48c7bb756d6652fb\",\n        \"glintr100.onnx\": \"a7933ea5330113b01c9b60351d8f4c33003f145d8470ac5f0e52ee2effe25c60\",\n        \"scrfd_10g_bnkps.onnx\": \"5838f7fe053675b1c7a08b633df49e7af5495cee0493c7dcf6697200b85b5b91\"\n      }\n    },\n    \"yolo\": {\n      \"repo\": \"Bingsu/adetailer\",\n      \"revision\": \"53cc19de382014514d9d4038601d261a7faa9b7b\",\n      \"filename\": \"face_yolov8n.pt\",\n      \"sha256\": \"70b640f8f60b1cf0dcc72f30caf3da9495eb2fb6509da48c53374ad6806e6a9c\"\n    }\n  },\n  \"pins\": {\n    \"numpy\": \"1.26.4\",\n    \"pandas\": \"2.2.2\",\n    \"Pillow\": \"11.3.0\",\n    \"matplotlib\": \"3.10.0\",\n    \"onnx\": \"1.17.0\",\n    \"onnxruntime\": \"1.19.2\",\n    \"insightface\": \"1.0.1\",\n    \"ultralytics\": \"8.4.121\",\n    \"opencv-python\": \"4.11.0.86\",\n    \"huggingface-hub\": \"0.36.2\",\n    \"scipy\": \"1.15.3\",\n    \"scikit-image\": \"0.25.2\",\n    \"packaging\": \"25.0\",\n    \"torch\": \"2.11.0+cpu\",\n    \"torchvision\": \"0.26.0+cpu\"\n  },\n  \"environment_boundary\": \"CPU-only private environment; same torch/torchvision release as N08, CPU builds replace cu128. YOLO bitwise GPU reproduction not presumed; saved crop/detection comparison reported.\",\n  \"package_source\": {\n    \"wheel_url\": \"https://files.pythonhosted.org/packages/42/ce/33989427a7fae952e1606d0edee4191f965630f3c74f2166fa776fafd271/insightface-1.0.1-py3-none-any.whl\",\n    \"wheel_sha256\": \"5f373f6fedbdda5cbc59a34ca386a75a2995cdaf6899402590ae9eb4308fc2e8\",\n    \"legacy_url\": \"https://files.pythonhosted.org/packages/0b/8d/0f4af90999ca96cf8cb846eb5ae27c5ef5b390f9c090dd19e4fa76364c13/insightface-0.7.3.tar.gz\",\n    \"legacy_sha256\": \"f191f719612ebb37018f41936814500544cd0f86e6fcd676c023f354c668ddf7\",\n    \"legacy_retinaface_sha256\": \"945597b54141a892419e0edfba464eb84fa2ec10ba800265c06ee274edc534ee\",\n    \"cpu_wheels\": {\n      \"torch\": {\n        \"url\": \"https://download-r2.pytorch.org/whl/cpu/torch-2.11.0%2Bcpu-cp312-cp312-manylinux_2_28_x86_64.whl\",\n        \"sha256\": \"f82e2ae20c1545bb03997d1cc3143d94e14b800038669ee1aca45808a9acc338\"\n      },\n      \"torchvision\": {\n        \"url\": \"https://download-r2.pytorch.org/whl/cpu/torchvision-0.26.0%2Bcpu-cp312-cp312-manylinux_2_28_x86_64.whl\",\n        \"sha256\": \"cf547dc0975eb40bc3249be4ccbeb736597d2c3ece305b1c4e5b7a5dd7363567\"\n      }\n    }\n  },\n  \"geometry_rules\": {\n    \"min_interocular_px\": 1,\n    \"landmark_box_tolerance_fraction\": 0.25,\n    \"alignment_condition_max\": 100000000,\n    \"warning_padding_fraction\": 0.5,\n    \"automatic_rules\": \"engineering plausibility only; no intended-face acceptance without actual manual review\"\n  },\n  \"privacy_similarity\": \"not_calculated\",\n  \"biometric_threshold\": null,\n  \"prompt\": \"not_applicable_existing_images\",\n  \"mask\": \"none\",\n  \"diffusion_parameters\": \"not_applicable\",\n  \"utility\": \"not_reevaluated\",\n  \"manual_review\": \"pending; never inferred from extraction\",\n  \"cpu_threads\": 2,\n  \"numeric_tolerance\": 0.0001,\n  \"input_contract\": {\n    \"prepared\": \"HWC uint8 BGR 0..255\",\n    \"blob\": \"NCHW float32 RGB (pixel-127.5)/128\",\n    \"pad_value\": 0,\n    \"coordinate_frame\": \"detector.detect returns prepared-input pixel coordinates; crop offsets then map to original\"\n  }\n}")
CONFIG_SHA256 = hashlib.sha256(json.dumps(CONFIG,sort_keys=True,separators=(",",":"),allow_nan=False).encode()).hexdigest()
RUNS_ROOT = Path("/content/drive/MyDrive/thesis_all/outputs/runs")
PARENT07 = RUNS_ROOT / CONFIG["parent07"]
PARENT08 = RUNS_ROOT / CONFIG["parent08"]
WORKER_PARTS = []
print("Frozen configuration:",CONFIG_SHA256,"; 14 sources; CPU diagnostic; no execution switches.")

CONFIG['variants']=[
 {'id':'V0_original_1024','canvas':1024,'kind':'direct','description':'Original full-frame resized by upstream SCRFD to 1024'},
 {'id':'V1_original_640','canvas':640,'kind':'direct','description':'Same source directly, upstream SCRFD resize to 640'},
 {'id':'V2_original_320','canvas':320,'kind':'direct','description':'Same source directly, upstream SCRFD resize to 320'},
 {'id':'V3_centered_640','canvas':640,'kind':'centered','description':'512 original isotropically reduced to 320 and centered on 640 black canvas'},
 {'id':'V4_centered_1024','canvas':1024,'kind':'centered1024','description':'512 original unresized, centered on black 1024 canvas'}]
CONFIG['method_id']='nb10_detector_scale_canvas_v1'
CONFIG['analysis']='detector_localization_only_no_embeddings_or_privacy'
CONFIG_SHA256=hashlib.sha256(json.dumps(CONFIG,sort_keys=True,separators=(',',':')).encode()).hexdigest()
print('NB10 frozen methods:', [(x['id'], x['canvas']) for x in CONFIG['variants']]);print('Config hash:',CONFIG_SHA256)


## 2. Reuse N09 verified private Python environment

The same pinned package set as N09 is verified automatically. The subprocess uses `MPLBACKEND=Agg`. No manual execution flags or runtime restart.

In [ ]:

if sys.platform != 'linux' or platform.machine() != 'x86_64' or sys.version_info[:2] != (3,12):
    raise RuntimeError('Notebook 10 needs Linux x86_64 CPython 3.12 (prefer 3.12.13). Select a compatible Colab runtime. No pins are substituted.')
libc_name,libc_version=platform.libc_ver()
if libc_name!='glibc' or tuple(map(int,libc_version.split('.')[:2]))<(2,28):
    raise RuntimeError('Pinned CPU wheels require glibc >=2.28; use compatible Colab Python 3.12 runtime.')
if platform.python_version() != '3.12.13':
    print('Runtime patch differs from N08:',platform.python_version(),'; recorded, never silently equated.')
from google.colab import drive
if not Path('/content/drive/MyDrive').is_dir(): drive.mount('/content/drive')

def sha_file(path):
    digest=hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda:handle.read(1024*1024),b''): digest.update(block)
    return digest.hexdigest()

PRIVATE_ENV=dict(os.environ)
for name in ['PYTHONPATH','PYTHONHOME','PIP_TARGET','PIP_USER','PIP_PREFIX']:
    PRIVATE_ENV.pop(name,None)
PRIVATE_ENV.update(PIP_CONFIG_FILE=os.devnull,CUDA_VISIBLE_DEVICES='',MPLBACKEND='Agg')

def run_process(command):
    print('Environment step:',command[1:4],flush=True)
    process=subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1,env=PRIVATE_ENV)
    for line in process.stdout: print(line.rstrip(),flush=True)
    if process.wait(): raise RuntimeError('Automatic isolated setup failed; no diagnostic inference started. See pip/import error above.')

pin_hash=hashlib.sha256(json.dumps(CONFIG['pins'],sort_keys=True).encode()).hexdigest()[:12]
ENV_DIR=Path('/content')/('nb09_env_'+platform.python_version()+'_'+pin_hash)
ENV_PY=ENV_DIR/'bin'/'python'
PROBE = r'''
import json,sys,platform,hashlib,inspect
from importlib import metadata
import numpy,pandas,PIL,matplotlib,onnx,onnxruntime,insightface,cv2,torch,torchvision,ultralytics
from insightface.app import FaceAnalysis
import insightface.model_zoo.scrfd as scrfd
import insightface.model_zoo.model_zoo as zoo
pins=json.loads(sys.argv[1])
for name,expected in pins.items():
    if metadata.version(name)!=expected: raise RuntimeError(f'Pin mismatch: {name}, expected {expected}')
if numpy.__version__!=pins['numpy'] or cv2.__version__!='.'.join(pins['opencv-python'].split('.')[:3]):
    raise RuntimeError('Native import/version collision')
if 'CPUExecutionProvider' not in onnxruntime.get_available_providers(): raise RuntimeError('CPU ONNX unavailable')
result={'python':platform.python_version(),'platform':platform.platform(),'packages':dict(sorted((d.metadata['Name'],d.version) for d in metadata.distributions())),
 'torch_device':str(torch.device('cpu')),'torch':torch.__version__,'torchvision':torchvision.__version__,
 'onnx_providers':onnxruntime.get_available_providers(),'source_sha256':{}}
for obj in [FaceAnalysis,scrfd.SCRFD,zoo.ModelRouter]:
    path=inspect.getsourcefile(obj)
    result['source_sha256'][obj.__name__]=hashlib.sha256(open(path,'rb').read()).hexdigest()
print('NB09_ENV '+json.dumps(result,sort_keys=True))
'''
# Colab may ship a Python interpreter whose ensurepip bootstrap cannot complete.
# Create a pip-free venv, then bootstrap pinned pip *inside that venv*
# using host pip's --python target (does not install packages into the host).
import venv
import shutil
PIP_BOOTSTRAP_VERSION = '24.3.1'

def private_pip_works():
    if not ENV_PY.is_file():
        return False
    try:
        check = subprocess.run(
            [str(ENV_PY), '-m', 'pip', '--version'],
            env=PRIVATE_ENV, text=True, capture_output=True, timeout=30,
        )
        return check.returncode == 0 and str(ENV_DIR) in check.stdout
    except (OSError, subprocess.TimeoutExpired):
        return False

if not private_pip_works():
    # Only the uniquely named, ephemeral /content/nb09_env_* directory is touched.
    if ENV_DIR.exists():
        if ENV_DIR.parent != Path('/content') or not ENV_DIR.name.startswith('nb09_env_'):
            raise RuntimeError('Unsafe private environment path; refusing cleanup')
        print('Cleaning incomplete Notebook 10 venv:', ENV_DIR, flush=True)
        shutil.rmtree(ENV_DIR)
    try:
        venv.EnvBuilder(with_pip=False, system_site_packages=False).create(ENV_DIR)
    except Exception as error:
        raise RuntimeError('Unable to create pip-free isolated venv: ' + repr(error)) from error
    # pip >=22.3 supports --python and can provision pip into a venv with no pip.
    host_pip = subprocess.run(
        [sys.executable, '-m', 'pip', '--version'],
        env=PRIVATE_ENV, text=True, capture_output=True, timeout=30,
    )
    if host_pip.returncode:
        raise RuntimeError('Host pip unavailable for target-only bootstrap: ' + host_pip.stderr[-3000:])
    run_process([
        sys.executable, '-m', 'pip', '--python', str(ENV_PY),
        '--isolated', 'install', '--disable-pip-version-check',
        '--index-url', 'https://pypi.org/simple',
        'pip==' + PIP_BOOTSTRAP_VERSION,
    ])
    if not private_pip_works():
        raise RuntimeError('Pinned pip bootstrap did not produce working private pip; no inference started')
VERSION_PROBE="import json,sys;from importlib.metadata import version; pins=json.loads(sys.argv[1]);assert all(version(k)==v for k,v in pins.items())"
versions_ready=subprocess.run([str(ENV_PY),'-c',VERSION_PROBE,json.dumps(CONFIG['pins'])],capture_output=True,env=PRIVATE_ENV).returncode==0
if not versions_ready:
    requirements=[name+'=='+version for name,version in CONFIG['pins'].items()]
    requirements=[value for value in requirements if not value.startswith(('insightface==','torch==','torchvision=='))]
    requirements += [spec['url']+'#sha256='+spec['sha256'] for spec in CONFIG['package_source']['cpu_wheels'].values()]
    requirements.append(CONFIG['package_source']['wheel_url']+'#sha256='+CONFIG['package_source']['wheel_sha256'])
    run_process([str(ENV_PY),'-m','pip','--isolated','install','--timeout','30','--retries','2',
                 '--index-url','https://pypi.org/simple',*requirements])
run_process([str(ENV_PY),'-m','pip','--isolated','check'])
probe_env=dict(PRIVATE_ENV,CUDA_VISIBLE_DEVICES='',YOLO_CONFIG_DIR=str(ENV_DIR/'yolo_settings'),MPLCONFIGDIR=str(ENV_DIR/'mpl'))
probe=subprocess.run([str(ENV_PY),'-c',PROBE,json.dumps(CONFIG['pins'])],env=probe_env,text=True,capture_output=True,timeout=180)
if probe.returncode: raise RuntimeError('Private environment import/ABI/pin preflight failed before inference: '+probe.stderr[-6000:])
probe_lines=[line[9:] for line in probe.stdout.splitlines() if line.startswith('NB09_ENV ')]
if len(probe_lines)!=1: raise RuntimeError('Environment probe missing structured output')
ENV_METADATA=json.loads(probe_lines[0])
EXPECTED_PACKAGE_SOURCE={'FaceAnalysis':'759271af722dafa1f602354550cdfd84cd705de0184a63b43752982635bcf5ff',
                         'SCRFD':'8898d1f90690d46bee371e1de31d1af60837988272a5eef904c5ba89e0fb8d83'}
for name,expected in EXPECTED_PACKAGE_SOURCE.items():
    if ENV_METADATA['source_sha256'][name]!=expected:raise RuntimeError('Installed official InsightFace source mismatch: '+name)
ENV_SHA256=hashlib.sha256(json.dumps(ENV_METADATA,sort_keys=True,separators=(',',':')).encode()).hexdigest()
print('Isolated environment verified; no restart/model loading. Fingerprint:',ENV_SHA256)


## 3. Read-only source and detector preflight

Validate frozen images, N09 output manifest, common SCRFD model identity, and saved YOLO reference boxes. New unique run directory.

In [ ]:
from PIL import Image
import json as _json
import hashlib as _hashlib
NB09_RUN = RUNS_ROOT/'20261013_nb09_detector-rootcause_4fae269d_3a2979d8_20261013T201108057643Z_02a058fe'
NB09_SESSION = NB09_RUN/'sessions'/'20261013T201108057643Z_02a058fe'
NB09_MANIFEST = NB09_SESSION/'session_manifest.json'
NB09_HASHES = NB09_SESSION/'model_file_hashes.json'
NB09_RECORDS = NB09_SESSION/'raw_records.json'
PARENT07 = RUNS_ROOT / CONFIG['parent07']
for p in [NB09_MANIFEST, NB09_HASHES, NB09_RECORDS]:
    if not p.is_file():raise FileNotFoundError('Preserved Notebook 09 artifact missing: '+str(p))
prev=_json.loads(NB09_MANIFEST.read_text())
if prev.get('manual_review_status')!='pending': print('N09 manual review status:',prev.get('manual_review_status'))
model_files=_json.loads(NB09_HASHES.read_text())
DETECTOR_HASH=CONFIG['models']['arcface']['onnx_sha256']['det_10g.onnx']
assert DETECTOR_HASH==CONFIG['models']['auraface']['onnx_sha256']['scrfd_10g_bnkps.onnx']
model_candidates=[Path(p) for p,h in model_files.items() if h==DETECTOR_HASH]
assert model_candidates,'No preserved verified detector identity in N09 manifest'
DETECTOR_PATH=next((p for p in model_candidates if p.is_file()),None)
if DETECTOR_PATH is None:raise FileNotFoundError('Previously verified SCRFD ONNX not in Drive model cache; do not silently swap weights')
if sha_file(DETECTOR_PATH)!=DETECTOR_HASH:raise RuntimeError('Detector changed since N09')
previous_records=_json.loads(NB09_RECORDS.read_text())
yolo_reference={}
for record in previous_records:
    if record['reference']=='R3' and record['evaluator']=='yolo_sanity':
        boxes=record.get('candidates',[])
        yolo_reference[record['source_id']]=boxes
assert len(yolo_reference)==14,'N09 YOLO reference incomplete'
source_paths={}
for s in CONFIG['sources']:
    p=PARENT07/'sources'/(s['source_id']+'.png')
    if not p.is_file() or sha_file(p)!=s['source_sha256']:raise RuntimeError('N07 source mismatch: '+s['source_id'])
    source_paths[s['source_id']]=str(p)
RUN_ID='nb10_scale_canvas_'+CONFIG_SHA256[:12]+'_'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')+'_'+uuid.uuid4().hex[:8]
OUT_DIR=RUNS_ROOT/RUN_ID
OUT_DIR.mkdir(exist_ok=False)
metadata={'config':CONFIG,'config_sha256':CONFIG_SHA256,'run_id':RUN_ID,'detector_path':str(DETECTOR_PATH),
 'detector_sha256':DETECTOR_HASH,'source_paths':source_paths,'yolo_reference':yolo_reference,'nb09_manifest_sha256':sha_file(NB09_MANIFEST),
 'nb09_raw_records_sha256':sha_file(NB09_RECORDS),'manual_review_status':'pending'}
(OUT_DIR/'config_and_context.json').write_text(_json.dumps(metadata,indent=2))
print('Frozen sample and N09 model verified; new isolated output:',OUT_DIR)


## 4. Automated detector-scale evaluation

Five variants × 14 sources = 70 detector-only records. Affine mappings preserve coordinates in original image space. Saved YOLO IoU is a comparison, not truth. No recognizer or embedding computation.

In [ ]:
import subprocess, textwrap
WORKER_SOURCE = 'import sys,os,json,hashlib,csv,traceback,time\nfrom pathlib import Path\nimport numpy as np,cv2\nfrom PIL import Image,ImageDraw\nfrom insightface.model_zoo import get_model\nctx=json.loads(Path(sys.argv[1]).read_text())\nroot=Path(sys.argv[2]); conf=ctx[\'config\']; variants=conf[\'variants\']; sha=ctx[\'detector_sha256\']\ndef digest(p):\n h=hashlib.sha256()\n with open(p,\'rb\') as f:\n  for b in iter(lambda:f.read(1024*1024),b\'\'):h.update(b)\n return h.hexdigest()\ndef json_write(p,x):\n with open(p,\'x\') as f:json.dump(x,f,indent=2,allow_nan=False)\ndef iou(a,b):\n if not a or not b:return None\n x1=max(a[0],b[0]);y1=max(a[1],b[1]);x2=min(a[2],b[2]);y2=min(a[3],b[3]);inter=max(0,x2-x1)*max(0,y2-y1)\n aa=max(0,a[2]-a[0])*max(0,a[3]-a[1]);bb=max(0,b[2]-b[0])*max(0,b[3]-b[1])\n return inter/(aa+bb-inter) if aa+bb-inter>0 else None\ndef make_variant(bgr,variant):\n h,w=bgr.shape[:2];kind=variant[\'kind\'];canvas=variant[\'canvas\']\n if kind==\'direct\':return bgr,np.eye(3),{\'transform\':\'identity\',\'detector_scales_original_to_canvas\':canvas/w}\n if kind==\'centered\':\n  size=320; resized=cv2.resize(bgr,(size,size),interpolation=cv2.INTER_AREA);ox=(canvas-size)//2;oy=(canvas-size)//2;scale=size/w\n else:\n  size=w;resized=bgr.copy();ox=(canvas-size)//2;oy=(canvas-size)//2;scale=1.0\n dst=np.zeros((canvas,canvas,3),dtype=np.uint8);dst[oy:oy+size,ox:ox+size]=resized\n T=np.array([[scale,0,ox],[0,scale,oy],[0,0,1]],float)\n return dst,T,{\'transform\':\'centered_on_black_canvas\',\'image_size_on_canvas\':size,\'offset_x\':ox,\'offset_y\':oy,\'scale\':scale}\ndef apply_inv(xy,T):\n z=np.asarray(xy,float).reshape(-1,2);h=np.concatenate([z,np.ones((len(z),1))],axis=1);v=h@np.linalg.inv(T).T;return v[:,:2]\ndef map_box(b,T):\n q=apply_inv([[b[0],b[1]],[b[2],b[3]]],T);return [float(q[0,0]),float(q[0,1]),float(q[1,0]),float(q[1,1])]\ndef annotated(img,boxes,ref):\n pic=Image.fromarray(cv2.cvtColor(img,cv2.COLOR_BGR2RGB));draw=ImageDraw.Draw(pic)\n for i,b in enumerate(boxes):\n  xy=b[\'bbox_source_xyxy\'];draw.rectangle(tuple(xy),outline=\'yellow\' if i==0 else \'cyan\',width=3)\n  draw.text((max(0,xy[0]),max(0,xy[1])),f"{i} {b[\'score\']:.3f}",fill=\'yellow\')\n for r in ref:\n  rb=r.get(\'bbox\');\n  if rb and len(rb)==4:draw.rectangle(tuple(rb),outline=\'magenta\',width=3)\n return pic\n\ndetector=Path(ctx[\'detector_path\'])\nif digest(detector)!=sha:raise RuntimeError(\'Detector SHA256 mismatch\')\nprint(\'Loading verified single shared SCRFD detector\',flush=True)\nmodel=get_model(str(detector),providers=[\'CPUExecutionProvider\'])\nif type(model).__name__!=\'SCRFD\':raise RuntimeError(\'Unexpected detector class\')\nnp.random.seed(42)\nallrecords=[]\nfor s in conf[\'sources\']:\n source_id=s[\'source_id\'];source_file=Path(ctx[\'source_paths\'][source_id]);assert digest(source_file)==s[\'source_sha256\']\n rgb=np.asarray(Image.open(source_file).convert(\'RGB\'));bgr=cv2.cvtColor(rgb,cv2.COLOR_RGB2BGR)\n references=ctx[\'yolo_reference\'].get(source_id,[])\n for variant in variants:\n  key=source_id+\'__\'+variant[\'id\'];out=root/(key+\'.json\');png=root/(key+\'.png\')\n  if out.exists():\n   rec=json.loads(out.read_text());allrecords.append(rec);continue\n  t=time.perf_counter();prepared,T,meta=make_variant(bgr,variant)\n  try:\n   size=variant[\'canvas\'];model.prepare(ctx_id=-1,input_size=(size,size),det_thresh=0.1)\n   boxes,kps=model.detect(prepared,max_num=0,metric=\'default\')\n   candidates=[]\n   for i,box in enumerate(boxes):\n    xy=map_box(box[:4],T)\n    landmarks=None if kps is None else apply_inv(kps[i],T).tolist()\n    matches=[iou(xy,r.get(\'bbox\')) for r in references if r.get(\'bbox\')]\n    candidates.append({\'bbox_source_xyxy\':xy,\'bbox_detector_xyxy\':[float(x) for x in box[:4]],\'score\':float(box[4]),\n      \'landmarks_source\':landmarks,\'max_iou_to_saved_yolo\':max((x for x in matches if x is not None),default=None)})\n   best=max((x[\'max_iou_to_saved_yolo\'] for x in candidates if x[\'max_iou_to_saved_yolo\'] is not None),default=None)\n   chosen=0 if candidates else None # Native highest-confidence order, not YOLO-oracle based\n   rec={\'source_id\':source_id,\'emotion\':s[\'emotion\'],\'stratum\':s[\'historical_stratum\'],\'variant\':variant[\'id\'],\n     \'canvas_size\':size,\'image_geometry\':meta,\'image_to_variant_matrix\':T.tolist(),\n     \'threshold\':0.1,\'num_detections\':len(candidates),\'selected_candidate_index\':chosen,\n     \'selected_confidence\':candidates[0][\'score\'] if candidates else None,\n     \'selected_iou_to_yolo\':candidates[0][\'max_iou_to_saved_yolo\'] if candidates else None,\n     \'best_candidate_iou_to_yolo\':best,\'candidates\':candidates,\'yolo_reference_count\':len(references),\n     \'manual_face_acceptance\':\'pending\',\'root_cause_proven\':False,\'runtime_seconds\':time.perf_counter()-t,\'error\':None}\n   annotated(bgr,candidates,references).save(png)\n  except Exception as e:\n   rec={\'source_id\':source_id,\'variant\':variant[\'id\'],\'num_detections\':None,\'candidates\':[],\n     \'manual_face_acceptance\':\'pending\',\'error\':type(e).__name__+\': \'+str(e),\'traceback\':traceback.format_exc()}\n  json_write(out,rec);allrecords.append(rec)\n  print(f"{len(allrecords)}/{len(conf[\'sources\'])*len(variants)} {key} detections={rec[\'num_detections\']}",flush=True)\nrows=[]\nfor r in allrecords:\n rows.append({k:r.get(k) for k in [\'source_id\',\'variant\',\'num_detections\',\'selected_confidence\',\'selected_iou_to_yolo\',\n  \'best_candidate_iou_to_yolo\',\'manual_face_acceptance\',\'error\']})\nwith open(root/\'records.csv\',\'x\',newline=\'\') as f:\n wr=csv.DictWriter(f,fieldnames=list(rows[0]));wr.writeheader();wr.writerows(rows)\nsummary=[]\nfor v in variants:\n rr=[r for r in allrecords if r[\'variant\']==v[\'id\']];match=[r for r in rr if r.get(\'selected_iou_to_yolo\') is not None]\n summary.append({\'variant\':v[\'id\'],\'n\':len(rr),\'detected\':sum((r.get(\'num_detections\') or 0)>0 for r in rr),\n  \'errors\':sum(r.get(\'error\') is not None for r in rr),\n  \'selected_mean_iou_to_yolo\':float(np.mean([r[\'selected_iou_to_yolo\'] for r in match])) if match else None,\n  \'manual_review\':\'pending\'})\njson_write(root/\'summary.json\',{\'status\':\'completed_pending_manual_review\',\'source_count\':len(conf[\'sources\']),\n \'planned_records\':len(conf[\'sources\'])*len(variants),\'completed_records\':len(allrecords),\n \'threshold\':0.1,\'variants\':summary,\'validated_face_count\':None,\'biometric_coverage\':None,\n \'notes\':\'YOLO is an independent approximation, not ground truth. IoU cannot establish validity. No embeddings or similarities calculated.\'})\nlines=[\'# Notebook 10 — detector scale / canvas diagnostic\',\'\',f"Completed {len(allrecords)} / {len(conf[\'sources\'])*len(variants)} cases.",\'\',\'## Variants\',\n \'| Variant | Detections >=1 | Errors | Mean selected IoU vs prior YOLO |\',\'|---|---:|---:|---:|\']\nfor x in summary:lines.append(f"| {x[\'variant\']} | {x[\'detected\']}/14 | {x[\'errors\']} | {x[\'selected_mean_iou_to_yolo\']} |")\nlines.extend([\'\',\'**Manual intended-face review: pending.**\',\'IoU to N09 YOLO boxes is an indirect agreement indicator, not ground truth.\',\n \'Models, 14 source images and old experiments were not changed.\',\'No privacy similarity, embeddings, or 600-output re-evaluation.\',\'Decision gate: CLOSED pending manual review.\'])\n(root/\'diagnostic_report.md\').write_text(\'\\n\'.join(lines)+\'\\n\')\nmanifest={p.name:digest(p) for p in root.iterdir() if p.is_file()}\njson_write(root/\'artifact_hashes.json\',manifest)\nprint(\'NB10_COMPLETED\',root,flush=True)\n'
worker_path=OUT_DIR/'nb10_worker.py'
worker_path.write_text(WORKER_SOURCE)
cmd=[str(ENV_PY),'-u',str(worker_path),str(OUT_DIR/'config_and_context.json'),str(OUT_DIR)]
process=subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,env=probe_env,bufsize=1)
for line in process.stdout: print(line.rstrip(),flush=True)
if process.wait()!=0:raise RuntimeError('Notebook 10 worker failed; preserved partial JSON results in '+str(OUT_DIR))


## 5. Results and manual-review boundary

Review the 70 labeled PNGs and `records.csv` in the new Google Drive run directory. The automatic report intentionally does not certify face validity.

In [ ]:
import json
summary=json.loads((OUT_DIR/'summary.json').read_text())
print(json.dumps(summary,indent=2))
print('REPORT:',OUT_DIR/'diagnostic_report.md')
print('ALL ARTIFACTS:',OUT_DIR)
assert summary['completed_records']==70
assert all(v['manual_review']=='pending' for v in summary['variants'])
